# GravLens-Net -- Phase 6: Redshift-Matched Negative Sample

**Goal:** Phase 5 fixed the galaxy-type confound (concentration) but found a
residual size/flux confound (AUC 0.79 from total flux alone), most likely
because the matched LRG catalog wasn't selected at the same *redshift* as
the SLACS lenses -- closer galaxies look bigger and brighter regardless of
type. This phase pulls each SLACS lens's actual spectroscopic redshift
(`zFG` column) and only keeps LRG-catalog candidates in that same redshift
range (via VizieR catalog V/139, which has a `zsp` spectroscopic redshift
column), on top of Phase 5's galaxy-type matching and position exclusion.


In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 131 real SLACS positives (Phase 4) + 167 real redshift-matched LRG negatives (Phase 6 fetch)
images = np.load("gravlens_phase6_images.npy")
labels = np.load("gravlens_phase6_labels.npy").astype(np.int32)
print(f"Phase 6 dataset: {images.shape}, positives={labels.sum()}/{len(labels)} "
      f"({100*labels.sum()/len(labels):.1f}%)")


I0000 00:00:1784693259.464054     797 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784693259.465149     797 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784693259.646331     797 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1784693261.149950     797 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784693261.151798     797 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Phase 6 dataset: (298, 101, 101), positives=131/298 (44.0%)


## 1. Re-check all three confound diagnostics

Concentration (Phase 4's confound), total flux, and half-light-radius
(Phase 5's residual confound) -- checking whether redshift matching
actually closed the size/flux gap.


In [2]:
def concentration(imgs, r_in=5, r_out=25):
    c = imgs.shape[1] // 2
    inner = imgs[:, c-r_in:c+r_in, c-r_in:c+r_in].mean(axis=(1, 2))
    outer = imgs[:, c-r_out:c+r_out, c-r_out:c+r_out].mean(axis=(1, 2))
    return inner / (outer + 1e-6)

def total_flux(imgs):
    return imgs.sum(axis=(1, 2))

def half_light_radius_proxy(imgs):
    c = imgs.shape[1] // 2
    y, x = np.mgrid[0:imgs.shape[1], 0:imgs.shape[2]]
    r = np.sqrt((x - c) ** 2 + (y - c) ** 2)
    order = np.argsort(r.ravel())
    r_sorted = r.ravel()[order]
    radii = []
    for img in imgs:
        img_pos = np.clip(img, 0, None)
        total = img_pos.sum()
        if total <= 0:
            radii.append(np.nan); continue
        cumsum = np.cumsum(img_pos.ravel()[order])
        idx = np.searchsorted(cumsum, 0.5 * total)
        radii.append(r_sorted[idx])
    return np.array(radii)

pos_imgs = images[labels == 1]
neg_imgs = images[labels == 0]

conc_auc = roc_auc_score(labels, concentration(images))
flux_auc = roc_auc_score(labels, total_flux(images))
hlr_all = half_light_radius_proxy(images)
valid = ~np.isnan(hlr_all)
hlr_auc = roc_auc_score(labels[valid], hlr_all[valid])

print(f"Concentration -- positives: {concentration(pos_imgs).mean():.2f}   negatives: {concentration(neg_imgs).mean():.2f}")
print(f"Total flux    -- positives: {total_flux(pos_imgs).mean():.2f}   negatives: {total_flux(neg_imgs).mean():.2f}")
print()
print(f"AUC concentration only:  {conc_auc:.3f}  (Phase 5 was 0.318)")
print(f"AUC total flux only:     {flux_auc:.3f}  (Phase 5 was 0.790)")
print(f"AUC half-light-radius:   {hlr_auc:.3f}  (Phase 5 was 0.319)")


Concentration -- positives: 9.88   negatives: 11.38
Total flux    -- positives: 260.51   negatives: 127.90

AUC concentration only:  0.362  (Phase 5 was 0.318)
AUC total flux only:     0.812  (Phase 5 was 0.790)
AUC half-light-radius:   0.442  (Phase 5 was 0.319)


**Mixed, honest result.** Half-light-radius (angular size) AUC moved from
0.319 to 0.442 -- much closer to 0.5 (no signal), confirming redshift
matching fixed the *angular size* confound as intended. But total flux AUC
actually got slightly *worse* (0.790 -> 0.812) and flipped direction
(negatives are now fainter, not brighter). Most likely explanation: SLACS
doesn't just select "any LRG at a given redshift" -- it specifically
requires high velocity dispersion (mass), which correlates with intrinsic
luminosity. So even redshift-matched, SLACS galaxies remain a
higher-luminosity subset than a general LRG catalog at the same redshift.
Fixing this fully would need matching on velocity dispersion too, not just
redshift -- an open item for a further phase.


## 2. Train the CNN on the redshift-matched dataset

In [3]:
def standardize(imgs):
    mean = imgs.mean(axis=(1, 2), keepdims=True)
    std = imgs.std(axis=(1, 2), keepdims=True) + 1e-6
    return (imgs - mean) / std

def augment_positives(X_pos, rng, factor=6):
    out = []
    for img in X_pos:
        out.append(img)
        for _ in range(factor - 1):
            k = rng.integers(0, 4)
            aug = np.rot90(img, k)
            if rng.random() < 0.5:
                aug = np.fliplr(aug)
            out.append(aug)
    return np.array(out)

def build_cnn(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(16, 3, padding="same"), layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, padding="same"), layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same"), layers.BatchNormalization(), layers.Activation("relu"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
                           tf.keras.metrics.Recall(name="recall"),
                           tf.keras.metrics.AUC(name="auc")])
    return model

images_std = standardize(images)
X_train, X_temp, y_train, y_temp = train_test_split(
    images_std, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)
print(f"train={len(y_train)} (pos={y_train.sum()})  val={len(y_val)} (pos={y_val.sum()})  test={len(y_test)} (pos={y_test.sum()})")

rng = np.random.default_rng(SEED)
pos_mask = y_train == 1
X_pos_aug = augment_positives(X_train[pos_mask], rng, factor=6)
X_train_aug = np.concatenate([X_train[~pos_mask], X_pos_aug], axis=0)
y_train_aug = np.concatenate([y_train[~pos_mask], np.ones(len(X_pos_aug), dtype=np.int32)])
perm = rng.permutation(len(X_train_aug))
X_train_aug, y_train_aug = X_train_aug[perm], y_train_aug[perm]

X_train_aug = X_train_aug[..., np.newaxis]
X_val_in = X_val[..., np.newaxis]
X_test_in = X_test[..., np.newaxis]

cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train_aug)
class_weight = {0: cw[0], 1: cw[1]}

model = build_cnn((images.shape[1], images.shape[2], 1))
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True)
model.fit(X_train_aug, y_train_aug, validation_data=(X_val_in, y_val),
          epochs=20, batch_size=32, class_weight=class_weight, verbose=2, callbacks=[early_stop])


train=208 (pos=91)  val=45 (pos=20)  test=45 (pos=20)


Epoch 1/20


E0000 00:00:1784693262.180326     797 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


21/21 - 6s - 308ms/step - accuracy: 0.7315 - auc: 0.7862 - loss: 0.5358 - precision: 0.9163 - recall: 0.7418 - val_accuracy: 0.6000 - val_auc: 0.7560 - val_loss: 0.6722 - val_precision: 0.5278 - val_recall: 0.9500


Epoch 2/20


...

21/21 - 5s - 252ms/step - accuracy: 0.8824 - auc: 0.9065 - loss: 0.3682 - precision: 0.9466 - recall: 0.9084 - val_accuracy: 0.5556 - val_auc: 0.8350 - val_loss: 0.9622 - val_precision: 0.5000 - val_recall: 1.0000


Epoch 16/20


21/21 - 5s - 231ms/step - accuracy: 0.8899 - auc: 0.9146 - loss: 0.3554 - precision: 0.9574 - recall: 0.9066 - val_accuracy: 0.7333 - val_auc: 0.8370 - val_loss: 0.5660 - val_precision: 0.6333 - val_recall: 0.9500


Epoch 17/20


21/21 - 5s - 243ms/step - accuracy: 0.8989 - auc: 0.9024 - loss: 0.3650 - precision: 0.9477 - recall: 0.9286 - val_accuracy: 0.7333 - val_auc: 0.8510 - val_loss: 0.5841 - val_precision: 0.6333 - val_recall: 0.9500


In [4]:
probs = model.predict(X_test_in, verbose=0).ravel()
preds = (probs >= 0.5).astype(int)
auc = roc_auc_score(y_test, probs)
precision = precision_score(y_test, preds, zero_division=0)
recall = recall_score(y_test, preds, zero_division=0)
f1 = f1_score(y_test, preds, zero_division=0)
cm = confusion_matrix(y_test, preds)

print("=== Phase 6: redshift-matched negatives ===")
print(f"ROC-AUC:   {auc:.3f}")
print(f"Precision: {precision:.3f}  Recall: {recall:.3f}  F1: {f1:.3f}")
print(f"Test set: n={len(y_test)}, positives={int(y_test.sum())}")
print("Confusion matrix [[TN,FP],[FN,TP]]:\n", cm)


=== Phase 6: redshift-matched negatives ===
ROC-AUC:   0.682
Precision: 0.513  Recall: 1.000  F1: 0.678
Test set: n=45, positives=20
Confusion matrix [[TN,FP],[FN,TP]]:
 [[ 6 19]
 [ 0 20]]


## 3. Results & Discussion -- the capstone finding of the real-data phases

| Phase | Negative sampling | ROC-AUC | Concentration AUC | Flux AUC | HLR AUC |
|---|---|---|---|---|---|
| Phase 4 | Random sky / random galaxies | 0.958-0.989 | ~1.0 (huge confound) | -- | -- |
| Phase 5 | Matched galaxy type (LRGs) | 0.956 | 0.32 (fixed) | 0.79 (confound) | 0.32 (confound) |
| **Phase 6** | + redshift-matched | **0.682** | 0.36 (still fixed) | 0.81 (persists, flipped) | 0.44 (mostly fixed) |

**This is the real headline of Phases 4-6 as a whole:** every time a
confound gets identified and fixed, the model's apparent performance drops
-- from ~0.99 to ~0.96 to **0.68**. That's not the model getting worse; it's
the evaluation getting *honest*. AUC 0.68 is only modestly better than
chance (0.5), and given the test set here is small (45 examples, 20
positives), even that number carries real uncertainty.

**What this means for the project overall:** the synthetic experiments in
Phase 3 predicted that real strong-lens detection would be a genuinely hard,
subtle-signal problem once realistic confusers were used (F1 collapsed from
0.857 to 0.057 there too, for similar reasons). Phases 4-6 now show the same
pattern with real observational data: naive real-data setups look deceptively
easy, and the true difficulty only becomes visible once the negative sample
is properly matched. Both the synthetic and real tracks of this project
arrived at the same conclusion through different paths, which is a
genuinely reassuring sign that the difficulty is real, not an artifact of
either track's specific choices.

**Remaining open item:** the residual flux confound (AUC 0.81) suggests
SLACS's velocity-dispersion selection isn't fully captured by redshift
matching alone. A Phase 7 would need to match on both redshift AND velocity
dispersion (or stellar mass/luminosity) simultaneously -- likely requiring
a catalog with spectroscopic velocity dispersion measurements, not just
photometric redshifts, which is a harder catalog to find via a generic
search.

**Practical note on sample size:** this whole phase runs on 131 positives
and a negative pool that shrinks every time matching gets stricter (1310 ->
422 -> 167). That's the real tradeoff of rigorous matching: a cleaner
comparison but less data and noisier metrics. Worth keeping in mind before
treating any single AUC number here as final.
